# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "DaiGreco", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_daigreco/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
# TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = None
from pyspark.sql import functions as F

# 1. Leer el lote nuevo desde challenge_path y la tabla existente bronze_events
challenge = spark.read.json(challenge_path)
bronze_events = spark.table(f"{config.catalog}.{config.schema}.bronze_events")

# 2. Comparación de campos para determinar cuál apareció
new_fields = set(challenge.columns) - set(bronze_events.columns)
print(f"1. Campo que apareció: {new_fields}")

# 3. Comparación de event_type para determinar cuál valor no existía
new_event_types = challenge.select("event_type").subtract(bronze_events.select("event_type"))
print("2. Valor de event_type que no existía:")
new_event_types.show()

# 4. Cálculo de filas totales e identificadores distintos en el lote nuevo
metrics = challenge.agg(
    F.count("*").alias("total_filas"),
    F.countDistinct("event_id").alias("distinct_ids")
)
print("3. Filas e identificadores distintos en el lote nuevo:")
display(metrics)

total_filas,distinct_ids
10001,10000


## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

# Leer histórico y nuevo lote
bronze = spark.table(source_table)
challenge = spark.read.json(challenge_path)

# Añadir app_version al histórico (NULL para eventos anteriores)
bronze = bronze.withColumn("app_version", F.lit(None).cast("string"))

# Añadir metadatos al lote nuevo
challenge = challenge.withColumn("_source", F.lit("challenge_v2")) \
    .withColumn("_source_file", F.lit("events_json_v2")) \
    .withColumn("_ingested_at", F.current_timestamp())

# Union por nombre admitiendo columnas faltantes y deduplicar por event_id
bronze_events_v2 = bronze.unionByName(challenge, allowMissingColumns=True) \
    .dropDuplicates(["event_id"])

# Escritura idempotente
bronze_events_v2.write.mode("overwrite").saveAsTable(target_table)

## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.table(target_table)
total_rows = result.count()
distinct_ids = result.select("event_id").distinct().count()
assert isinstance(total_rows, int) and isinstance(distinct_ids, int), "Completá los conteos: no pueden quedar en None"
assert total_rows == result.count(), "total_rows no coincide con la tabla"
assert distinct_ids == result.select("event_id").distinct().count(), "distinct_ids no coincide con la tabla"
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.

Detectar una evolución es descrubir los datos entrantes, si hay campos nuevo eliminados o modificados. Asi como que tipo de modificaciones se hicieron.

Aceptarla técnicamente significa que la infraestructura actual soporta el cambio. Que puede adaptarse a los cambios sin perder datos importantes ni romperse.

Decidir que es válida para el negocio implica comprobar que la modificación responde a una lógica de negocio real. Garantiza que los modelos, reportes y métricas sigan siendo confiables antes de propagar los datos.
